<a href="https://colab.research.google.com/github/JxseDiaz/Deep-Learning/blob/main/EP1_DLY0100_Castillo_Diaz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación Parcial N°1 – Fundamentos de Deep Learning (DLY0100)
**Integrantes:** Victoria Castillo y José Díaz  
**Caso:** Predicción de abandono de clientes (Telco Customer Churn) con un MLP en TensorFlow/Keras

## 1. Introducción
Una empresa de telecomunicaciones pierde ingresos cada vez que un cliente se va. Si podemos anticipar quién tiene más probabilidad de irse, el área comercial puede ofrecerle un plan o descuento a tiempo.

**Problema:** clasificación binaria (`Churn`: 1 = el cliente se va, 0 = se queda).  
**Objetivo:** construir un MLP que detecte a los clientes que abandonan, priorizando el **recall** de la clase "se va" (que no se nos escapen clientes en riesgo) sin que la precisión caiga demasiado.  
**Cómo vamos a trabajar:** preprocesar los datos, entrenar un modelo base, hacer experimentos variando un parámetro a la vez, comparar funciones de activación y error, probar regularización, y evaluar el modelo final con el set de test.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)
plt.rcParams['figure.figsize'] = (11, 4)
pd.set_option('display.float_format', '{:.3f}'.format)

## 2. Carga y preprocesamiento de datos
Cargamos el CSV del dataset Telco Customer Churn (7.043 clientes, 21 columnas). Si la descarga automática falla, el código pide subir el archivo manualmente.

In [ ]:
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
try:
    df = pd.read_csv(URL)
except Exception:
    from google.colab import files
    nombre = list(files.upload().keys())[0]   # subir WA_Fn-UseC_-Telco-Customer-Churn.csv
    df = pd.read_csv(nombre)

print(df.shape)
print(df['Churn'].value_counts(normalize=True))
df.head(3)

**Decisiones de preprocesamiento (y por qué):**
1. **Eliminar `customerID`:** es un identificador único, no aporta información y solo agregaría ruido.
2. **`TotalCharges` a numérico:** viene como texto porque tiene 11 espacios en blanco (clientes nuevos con `tenure = 0`). Los convertimos a número y los rellenamos con 0, porque todavía no han pagado nada.
3. **`Churn` a 0/1:** la red necesita una salida numérica.
4. **One-hot encoding** de las variables categóricas (`drop_first=True` para no duplicar información): una red neuronal no puede trabajar con texto.
5. **Partición estratificada 70/15/15** (train/validación/test): la validación sirve para ajustar y el test se usa solo una vez al final. Estratificamos para mantener la proporción de ~27% de churn en los tres conjuntos.
6. **Estandarización (`StandardScaler`)** ajustada solo con train, para evitar fuga de información. Sin esto, variables como `TotalCharges` (miles) dominarían sobre las binarias (0/1) y el entrenamiento sería inestable.
7. **Pesos de clase:** como solo ~27% de los clientes se va, un modelo podría ignorar esa clase y aun así tener buen accuracy. Con `class_weight` el error en la clase minoritaria pesa más.

In [ ]:
df = df.drop(columns='customerID')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = pd.get_dummies(df, drop_first=True, dtype=int)

X = df.drop(columns='Churn').values
y = df['Churn'].values

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

pesos = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
cw = {0: pesos[0], 1: pesos[1]}

print('Variables de entrada:', X_train.shape[1])
print('Train / Val / Test:', X_train.shape[0], X_val.shape[0], X_test.shape[0])
print('Pesos de clase:', cw)

## 3. Definición del modelo
Armamos dos funciones para no repetir código: `crear_modelo` (arma el MLP con los hiperparámetros que le pasemos) y `entrenar` (entrena, calcula métricas en validación y guarda el resultado en una tabla general).

**Modelo base:** `Input(n) → Dense(64, relu) → Dense(32, relu) → Dense(1, sigmoid)`
- **ReLU en capas ocultas:** barata de calcular y evita en gran parte el desvanecimiento del gradiente.
- **Sigmoid en la salida:** entrega una probabilidad entre 0 y 1, que es lo que necesitamos en clasificación binaria.
- **Binary crossentropy:** es la pérdida natural para una salida sigmoide, penaliza fuerte las predicciones seguras pero equivocadas.
- **Adam (lr = 0.001, batch = 32, 30 épocas):** punto de partida estándar; después lo ajustamos con experimentos.

In [ ]:
resultados = []   # aquí se acumulan todas las pruebas para la tabla comparativa final

def metricas(y_real, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    return {'accuracy': accuracy_score(y_real, pred), 'precision': precision_score(y_real, pred),
            'recall': recall_score(y_real, pred), 'f1': f1_score(y_real, pred)}

def crear_modelo(capas=(64, 32), act='relu', lr=1e-3, dropout=0.0, l2=0.0, bn=False, loss='binary_crossentropy'):
    reg = regularizers.l2(l2) if l2 > 0 else None
    m = keras.Sequential([layers.Input(shape=(X_train.shape[1],))])
    for u in capas:
        m.add(layers.Dense(u, activation=None if bn else act, kernel_regularizer=reg))
        if bn:
            m.add(layers.BatchNormalization())
            m.add(layers.Activation(act))
        if dropout > 0:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=['accuracy'])
    return m

def entrenar(nombre, epocas=30, batch=32, callbacks=None, **kw):
    tf.keras.utils.set_random_seed(SEED)
    m = crear_modelo(**kw)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epocas, batch_size=batch,
              class_weight=cw, callbacks=callbacks, verbose=0)
    prob = m.predict(X_val, verbose=0).ravel()
    resultados.append({'prueba': nombre, **metricas(y_val, prob), 'épocas': len(h.history['loss'])})
    return m, h

def curvas(hist, titulo):
    fig, ax = plt.subplots(1, 2)
    for nombre, h in hist.items():
        ax[0].plot(h.history['val_loss'], label=nombre)
        ax[1].plot(h.history['val_accuracy'], label=nombre)
    ax[0].set_title(titulo + ' – loss de validación'); ax[1].set_title(titulo + ' – accuracy de validación')
    for a in ax: a.set_xlabel('época'); a.legend()
    plt.tight_layout(); plt.show()

def tabla(prefijo):
    t = pd.DataFrame(resultados)
    return t[t['prueba'].str.startswith(prefijo)].set_index('prueba')

crear_modelo().summary()

## 4. Entrenamiento y ajuste de hiperparámetros (experimentos controlados)
En cada experimento cambiamos **un solo parámetro** y dejamos el resto igual al modelo base. Todos los modelos usan la misma semilla para que la comparación sea justa.

### 4.1 Épocas (10, 30 y 80)

In [ ]:
hist = {}
for ep in [10, 30, 80]:
    _, hist[f'{ep} épocas'] = entrenar(f'Épocas={ep}', epocas=ep)
curvas(hist, 'Épocas')
tabla('Épocas')

**Qué vemos:** las tres corridas usan la misma semilla, por eso las curvas se superponen: es el mismo entrenamiento visto en distintos puntos. La pérdida de validación baja hasta las ~8 épocas y después empieza a subir, mientras el accuracy casi no cambia (~0.73). Eso indica sobreajuste: el modelo sigue mejorando en train pero memoriza. En la tabla se nota en el recall, que baja de 0.80 (10 épocas) a 0.66 (80 épocas), y el F1 de 0.62 a 0.57.

**Decisión:** no vale la pena entrenar muchas épocas. Más adelante dejamos que `EarlyStopping` elija el momento justo.

### 4.2 Tasa de aprendizaje (0.0001, 0.001 y 0.01)

In [ ]:
hist = {}
for lr in [0.0001, 0.001, 0.01]:
    _, hist[f'lr={lr}'] = entrenar(f'LR={lr}', lr=lr)
curvas(hist, 'Learning rate')
tabla('LR')

**Qué vemos:** con `lr = 0.01` la pérdida de validación es muy inestable y termina subiendo fuerte (el optimizador da pasos demasiado grandes). Con `lr = 0.0001` la curva baja suave pero lenta, y con `0.001` converge rápido y estable. El accuracy de `0.01` parece mejor (0.76), pero es engañoso: su recall cae a 0.64, es decir, deja pasar más clientes que se van.

**Decisión:** nos quedamos con `lr = 0.001`, que equilibra velocidad y estabilidad.

### 4.3 Tamaño de batch (16, 64 y 256)

In [ ]:
hist = {}
for b in [16, 64, 256]:
    _, hist[f'batch={b}'] = entrenar(f'Batch={b}', batch=b)
curvas(hist, 'Batch size')
tabla('Batch')

**Qué vemos:** las diferencias entre batch 16, 64 y 256 son pequeñas (F1 entre 0.60 y 0.62). Con batch chico las curvas son más ruidosas y cada época tarda más; con batch grande es más estable pero hace menos actualizaciones de pesos.

**Decisión:** usamos `batch = 64`, que dio un F1 levemente mejor y es un punto medio entre ruido y tiempo de cómputo.

### 4.4 Número de capas y neuronas
Probamos tres arquitecturas con el resto de los parámetros fijos.

In [ ]:
hist = {}
for nombre, capas in {'1 capa (32)': (32,), '2 capas (64-32)': (64, 32), '3 capas (128-64-32)': (128, 64, 32)}.items():
    _, hist[nombre] = entrenar(f'Capas={nombre}', capas=capas)
curvas(hist, 'Arquitectura')
tabla('Capas')

**Qué vemos:** a más capas, menor recall (0.81 con 1 capa, 0.74 con 2 y 0.62 con 3) y menor F1. El dataset es pequeño (~5.000 filas de entrenamiento) y de tipo tabular, así que una red más grande se sobreajusta más rápido sin ganar precisión.

**Decisión:** mantenemos 2 capas (64-32). La de 1 capa rinde parecido, pero con 2 capas tenemos más capacidad y la regularización se encarga de controlar el sobreajuste. Ojo: estas diferencias son de pocos puntos y con una sola semilla, así que las tomamos como tendencia y no como algo exacto.

## 5. Funciones de activación y de error
Comparamos tres activaciones para las capas ocultas (ReLU, tanh y sigmoid) y dos funciones de error (binary crossentropy y MSE). La salida se mantiene en sigmoid porque necesitamos una probabilidad.

In [ ]:
hist = {}
for a in ['relu', 'tanh', 'sigmoid']:
    _, hist[f'{a} + BCE'] = entrenar(f'Func={a} + BCE', act=a)
_, hist['relu + MSE'] = entrenar('Func=relu + MSE', loss='mean_squared_error')
curvas(hist, 'Activación / error')
tabla('Func')

**Qué vemos:** las tres activaciones quedan muy cerca (F1 entre 0.60 y 0.63; sigmoid y tanh un poco arriba de ReLU en esta corrida). Sobre el error, BCE y MSE dan resultados parecidos aquí, pero BCE es la opción coherente con una salida sigmoide: su gradiente es mayor cuando la predicción está muy equivocada, mientras que MSE aprende más lento en esos casos.

**Decisión y validación:** dejamos **ReLU** en capas ocultas (es la estándar en MLP y no sufre desvanecimiento del gradiente si la red crece), **sigmoid** en la salida (probabilidad de churn) y **binary crossentropy** como pérdida, que son las funciones que corresponden a un problema de clasificación binaria. Como la diferencia con tanh/sigmoid es chica, esta elección se apoya más en el fundamento teórico que en el resultado numérico.

## 6. Optimización y regularización
Comparamos el modelo base contra distintas técnicas para controlar el sobreajuste: **Dropout (0.3)**, **regularización L2 (0.001)**, **Batch Normalization** y la combinación de las tres. Aquí graficamos train vs validación para ver el sobreajuste.

In [ ]:
configs = {'Sin regularización': {}, 'Dropout 0.3': {'dropout': 0.3}, 'L2 0.001': {'l2': 1e-3},
           'BatchNorm': {'bn': True}, 'Dropout+L2+BN': {'dropout': 0.3, 'l2': 1e-3, 'bn': True}}
hist = {}
for nombre, kw in configs.items():
    _, hist[nombre] = entrenar(f'Reg={nombre}', epocas=50, **kw)

fig, ax = plt.subplots(1, len(hist), figsize=(18, 3.2), sharey=True)
for a, (n, h) in zip(ax, hist.items()):
    a.plot(h.history['loss'], label='train'); a.plot(h.history['val_loss'], label='val'); a.set_title(n); a.set_xlabel('época')
ax[0].set_ylabel('loss'); ax[0].legend(); plt.tight_layout(); plt.show()
tabla('Reg')

**Qué vemos:** sin regularización la pérdida de validación sube desde el inicio mientras la de train sigue bajando (sobreajuste claro). `BatchNorm` solo empeora la brecha y el recall (0.58). `L2` ayuda poco. **Dropout 0.3** es lo que mejor estabiliza: la curva de validación queda plana y cerca de la de train, y da el mejor F1 (0.63) y un recall de 0.80. La combinación Dropout+L2+BN es estable, pero no mejora a Dropout solo.

**Decisión:** usamos solo Dropout 0.3. Para este tamaño de red y de datos, agregar más técnicas no aportó.

## 7. Modelo final
Con lo observado en los experimentos definimos la configuración final. Usamos **EarlyStopping** (para en el mejor punto de validación) para no depender de elegir a mano el número de épocas.

**Configuración elegida y por qué:**

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| Capas | 64 → 32 → 1 | Capacidad suficiente sin sobreajustar tanto como la de 3 capas |
| Activación oculta / salida | ReLU / Sigmoid | Estándar para clasificación binaria |
| Pérdida | Binary crossentropy | Coherente con salida sigmoide |
| Optimizador y lr | Adam, 0.001 | Convergencia estable (0.01 oscila, 0.0001 es lento) |
| Batch | 64 | F1 levemente mejor y buen tiempo de cómputo |
| Regularización | Dropout 0.3 | Mejor estabilidad en validación |
| Épocas | `EarlyStopping` (paciencia 8) | La validación empeora después de ~10 épocas |

In [ ]:
CONFIG = dict(capas=(64, 32), act='relu', lr=1e-3, dropout=0.3, l2=0.0, bn=False)
BATCH = 64

es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
modelo, h_final = entrenar('FINAL', epocas=150, batch=BATCH, callbacks=[es], **CONFIG)
print('Épocas entrenadas:', len(h_final.history['loss']))
tabla('FINAL')

## 8. Evaluación del modelo final (set de test)
Ahora sí usamos el test, que el modelo nunca vio. Calculamos las métricas y las comparamos con las de validación.

In [ ]:
prob_test = modelo.predict(X_test, verbose=0).ravel()
pred_test = (prob_test >= 0.5).astype(int)
m = metricas(y_test, prob_test)
cuadro = pd.DataFrame({'Test': m}).T
cuadro

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}\n')
print(f'Accuracy  = (TP+TN)/total = ({tp}+{tn})/{tn+fp+fn+tp} = {(tp+tn)/(tn+fp+fn+tp):.3f}')
print(f'Precision = TP/(TP+FP)    = {tp}/({tp}+{fp}) = {tp/(tp+fp):.3f}')
print(f'Recall    = TP/(TP+FN)    = {tp}/({tp}+{fn}) = {tp/(tp+fn):.3f}')
print(f'F1        = 2·P·R/(P+R)   = {2*tp/(2*tp+fp+fn):.3f}')
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=['Se queda', 'Se va']).plot(cmap='Blues')
plt.show()

**Interpretación:** en test el modelo obtiene accuracy ≈ 0.75, precision ≈ 0.52, recall ≈ 0.78 y F1 ≈ 0.63, muy parecido a lo que vimos en validación (F1 ≈ 0.62), así que generaliza bien y no se sobreajustó al conjunto de validación.

- **Recall (0.78):** de los clientes que realmente se van, el modelo detecta ~8 de cada 10 (220 de 281). Es la métrica que más nos importa en este caso.
- **Precision (0.52):** de los clientes que el modelo marca como "se va", solo la mitad realmente se va (202 falsas alarmas). Es el costo de priorizar el recall (y de haber usado pesos de clase).
- **Accuracy (0.75):** es la menos informativa aquí, porque con 73% de clientes que se quedan un modelo que dijera siempre "se queda" ya tendría ~0.73.
- **F1 (0.63):** resume el equilibrio entre precision y recall.

Una falsa alarma cuesta poco (ofrecer un descuento a quien igual se quedaba), mientras que un cliente perdido sin detectar cuesta bastante más. Por eso aceptamos una precision moderada a cambio de un recall alto.

### Cómo mejorar precision y recall: ajuste del umbral
El umbral de 0.5 no es sagrado. Lo movemos para ver el equilibrio entre precision y recall (usando validación para no hacer trampa con el test).

In [ ]:
prob_val = modelo.predict(X_val, verbose=0).ravel()
filas = [{'umbral': u, **metricas(y_val, prob_val, u)} for u in [0.3, 0.4, 0.5, 0.6, 0.7]]
tab_umbral = pd.DataFrame(filas).set_index('umbral')
display(tab_umbral)
tab_umbral[['precision', 'recall', 'f1']].plot(marker='o', title='Precision / Recall / F1 según umbral (validación)')
plt.show()

**Cómo mejorar precision o recall:** al subir el umbral el modelo es más exigente, la precision sube (0.50 → 0.62 entre 0.5 y 0.7) pero el recall cae (0.82 → 0.59). Al bajarlo pasa lo contrario: con 0.3 detecta el 95% de los que se van, pero con precision de 0.41. El mejor F1 en validación se da cerca de 0.6, aunque la diferencia con 0.5 es mínima.

Mantenemos **0.5** porque el negocio prefiere no perder clientes; si el presupuesto para retención fuera limitado, convendría subir el umbral hacia 0.6 para contactar menos clientes con mejor puntería. Otras mejoras posibles serían probar más variables derivadas (por ejemplo, cantidad de servicios contratados) y validación cruzada para confirmar que los resultados no dependen de una sola partición.

### Tabla comparativa de todas las pruebas (validación)

In [ ]:
pd.DataFrame(resultados).set_index('prueba').sort_values('f1', ascending=False)

## 9. Conclusiones
- **Preprocesamiento:** limpiar `TotalCharges`, codificar las categóricas, estandarizar y usar pesos de clase fue necesario para que la red aprendiera la clase minoritaria (churn, ~27%) y no se quedara con predecir siempre "se queda".
- **Hiperparámetros:** la tasa de aprendizaje fue lo que más influyó en la estabilidad (0.01 oscila, 0.001 funciona bien). El batch casi no cambió los resultados, y más épocas o más capas solo trajeron sobreajuste.
- **Funciones:** ReLU + sigmoid + binary crossentropy son las coherentes con el problema. Las diferencias numéricas con tanh/sigmoid y MSE fueron pequeñas, así que la elección se justifica más por teoría que por una gran ventaja medida.
- **Regularización:** Dropout 0.3 fue la técnica que más estabilizó el entrenamiento. Batch Normalization y L2 no aportaron en este dataset, que es chico y tabular.
- **Resultado final (test):** accuracy 0.75, precision 0.52, recall 0.78 y F1 0.63. El modelo detecta la mayoría de los clientes en riesgo, a costa de varias falsas alarmas.
- **Limitaciones y mejoras:** las diferencias entre muchas pruebas son de pocos puntos y se hicieron con una sola semilla, por lo que habría que repetirlas con validación cruzada. También se podrían probar nuevas variables, comparar con un modelo más simple (regresión logística) y ajustar el umbral según el costo real de cada tipo de error.